# Quantization Profiler — Colab test drive

Tests `Quantizer` end to end: quantize TinyLlama to W4A16, then serve the result with
`VLLMServerManager` and measure the VRAM saving against fp16.

**Runtime:** set *Runtime → Change runtime type → T4 GPU* (free tier is enough).

**Section A** (quantize) is light and fast. **Section B** (serve) installs vLLM, which is
multi-GB and replaces Colab's torch — do A first and confirm it works before paying for B.

Scheme support by GPU, since it decides what you can run here:

| Scheme | T4 (CC 7.5) | A100 (8.0) | L4 / H100 (8.9+) |
|---|---|---|---|
| `W4A16` | yes | yes | yes |
| `W8A8_INT8` | yes | yes | yes |
| `AWQ_W4A16` | yes | yes | yes |
| `FP8_BLOCK` | **no** | **no** | yes |

`W4A16` serving needs compute capability 75+ (`CompressedTensorsWNA16.get_min_capability()`),
so a T4 just clears it. FP8 needs 89+.

## 0. Confirm the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
major, minor = torch.cuda.get_device_capability()
print(f"{torch.cuda.get_device_name(0)}  compute capability {major}.{minor}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("FP8_BLOCK usable:", (major, minor) >= (8, 9))

## 1. Get the code

If the repo is private, either make it public, use
`https://<token>@github.com/...`, or just upload `src/` via the Files panel.

In [ ]:
!git clone --branch quantizer https://github.com/lelouch0204/quantization-profiler.git 2>/dev/null || echo "already cloned"

import sys

sys.path.insert(0, "/content/quantization-profiler/src")

## 2. Unit tests first — no GPU, no ML stack, 1 second

Cheapest possible signal that the code is intact before spending GPU time on it.

In [ ]:
!cd /content/quantization-profiler && python -m pytest tests/ -q

# Section A — Quantization

`llmcompressor` only, with **torch constrained to Colab's version**.

This constraint is the single most important line in the notebook. Letting pip upgrade
torch in place produces a half-overwritten package tree that fails on import with
`TypeError: Config() got an unexpected keyword argument 'deprecated'` and can only be
recovered by *Runtime → Disconnect and delete runtime*. Never use
`--upgrade-strategy eager` here.

If you are recovering from that state now: delete the runtime, then start again from cell 0.

In [ ]:
import torch

# Colab ships a working torch. pip must never overwrite it in place: doing so leaves a
# mixed package tree (a new torch/_dynamo/config.py against an old
# torch/utils/_config_module.py), which dies on import with
#   TypeError: Config() got an unexpected keyword argument 'deprecated'
# and is only recoverable by deleting the runtime. Constraining torch to the installed
# version makes pip fail loudly on a genuine conflict instead of corrupting the install.
with open("/content/constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")

print(f"pinning torch=={torch.__version__}")

# llmcompressor 0.13.0+ supports torch 2.13 (Colab Aug 2026 runtime).
# Do NOT add --upgrade-strategy eager: it drags torch forward and corrupts the runtime.
!pip install -q -c /content/constraints.txt "llmcompressor>=0.13.0"


In [ ]:
# Verify the install before spending GPU minutes on it.
#
# Ignore pip conflict warnings about protobuf, opentelemetry, numba, cudf/cuml or
# jedi -- those are Colab's preinstalled packages and are not in our import path.
# A compressed-tensors mismatch is the one that matters: llmcompressor pins it with
# ==, so a mismatch means "Runtime -> Restart session", then re-run this cell.
import importlib.metadata as md

for pkg in ("llmcompressor", "compressed-tensors", "transformers", "datasets", "torch"):
    try:
        print(f"{pkg:20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:20} MISSING")

pin = next(
    (r for r in md.requires("llmcompressor") or [] if r.startswith("compressed-tensors")),
    None,
)
print(f"\nllmcompressor requires: {pin or 'no compressed-tensors pin found'}")

# The real check: do the imports this project actually uses resolve? A version skew
# that matters shows up here as an ImportError, which is the signal to restart.
from llmcompressor import oneshot
from llmcompressor.modifiers.gptq import GPTQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform import AWQModifier, SmoothQuantModifier

print("all recipe imports OK")

Quantize TinyLlama-1.1B to W4A16. `num_samples=8` keeps this to a few minutes; the
default 512 is what you'd use for a real run.

`output_root` is under `/content`, which does **not** survive a runtime restart. Point it at
`/content/drive/MyDrive/...` after mounting Drive if you want the cache to persist.

In [ ]:
from quantizer import CalibrationConfig, QuantScheme, Quantizer

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

q = Quantizer(
    output_root="/content/quantized_models",
    calibration=CalibrationConfig(num_samples=8, max_seq_length=512),
)

# Where each scheme would land -- pure hashing, runs instantly, no GPU touched.
for scheme in QuantScheme:
    print(f"{scheme.name:12} -> {q.output_dir(MODEL, scheme).name}")

In [ ]:
path = q.quantize(MODEL, QuantScheme.W4A16)
print("checkpoint:", path)

### Check what came out

The point of `compressed-tensors` is that the checkpoint describes its own quantization in
`config.json` — which is why vLLM needs no `--quantization` flag.

In [ ]:
import json
import subprocess

print(subprocess.run(["du", "-sh", str(path)], capture_output=True, text=True).stdout)
print(sorted(p.name for p in path.iterdir()))

cfg = json.loads((path / "config.json").read_text())
print("\nquantization_config:")
print(json.dumps(cfg.get("quantization_config", {}), indent=2)[:900])

print("\nquant_key.json (which knobs produced this dir):")
print((path / "quant_key.json").read_text())

### Cache behaviour

Second call must return the same path instantly. Changing calibration must produce a
*different* path — that's the hash-keyed cache doing its job.

In [ ]:
import time

start = time.monotonic()
again = q.quantize(MODEL, QuantScheme.W4A16)
print(f"cache hit in {time.monotonic() - start:.3f}s -> {again == path}")

retuned = Quantizer(
    output_root="/content/quantized_models",
    calibration=CalibrationConfig(num_samples=64, max_seq_length=512),
)
print("different calibration -> different dir:", retuned.output_dir(MODEL, QuantScheme.W4A16).name)

try:
    q.quantize(MODEL, QuantScheme.NVFP4)
except NotImplementedError as e:
    print("placeholder scheme:", e)

# Section B — Serving

**Installs vLLM (multi-GB) and will likely replace torch, requiring a runtime restart.**
After restarting, re-run cells 0 and 1, then skip straight to here — the checkpoint from
Section A is still on disk, so `quantize()` would be a cache hit anyway.

On a free T4 this is the tightest part of the notebook: vLLM preallocates ~90% of VRAM by
default, so `gpu_memory_utilization` is worth lowering if you hit OOM.

In [ ]:
# vLLM pins an exact torch build, so this is the one place a torch change is expected.
# Installing it WITHOUT the constraint file is deliberate -- but let pip replace torch
# wholesale, then restart the runtime before importing anything, so no half-loaded
# torch modules linger in memory.
!pip install -q vllm

print("\nIf torch changed above: Runtime -> Restart session, then re-run cells 0-1 and")
print("skip straight back here. The Section A checkpoint is still on disk.")

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, "/content/quantization-profiler/src")

import requests
from vllm_server_manager import VLLMServerManager

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Pick up the checkpoint Section A produced -- it survives a runtime restart.
ckpt = next(p for p in Path("/content/quantized_models").iterdir() if "W4A16" in p.name)
print("serving:", ckpt)

vLLM's startup logs stream into this cell's output. Loading takes a couple of minutes; a
`wait_for_health` timeout almost always means the log above says why (usually OOM).

In [ ]:
manager = VLLMServerManager()
manager.start_server(str(ckpt), port=8000)
manager.wait_for_health(600)
print("healthy -- served with no --quantization flag")

In [ ]:
resp = requests.post(
    "http://localhost:8000/v1/completions",
    json={"model": str(ckpt), "prompt": "Quantization is", "max_tokens": 40},
    timeout=60,
)
print(resp.json()["choices"][0]["text"])

In [ ]:
def vram():
    """Total VRAM in use on GPU 0, per nvidia-smi."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    return out.stdout.strip()


# The number the profiler exists to produce.
print("W4A16 loaded:", vram())

In [ ]:
manager.terminate()
print("terminated, VRAM released:", vram())

### fp16 baseline

Needs no quantization at all — serve the hub id directly. The delta against the cell above
is your first real datapoint.

Note that vLLM preallocates a KV-cache block by `gpu_memory_utilization`, so `nvidia-smi`
reflects *reserved* memory, not weights alone. To compare weights honestly, either read the
"model weights take X GB" line in vLLM's own startup log, or just compare `du -sh` on the
checkpoints.

In [ ]:
baseline = VLLMServerManager()
try:
    baseline.start_server(MODEL, port=8001)
    baseline.wait_for_health(600)
    print("fp16 baseline:", vram())
finally:
    baseline.terminate()